# 01. Xác minh các bộ NetFlow v3 (task 1.2)

Chạy trên Kaggle, gắn dataset `athena21/netflow-v3-datasets`. Link tải chính thức của UQ (`rdm.uq.edu.au`) yêu cầu đăng nhập (API trả 401), nên dùng bản đăng lại này và **đối chiếu số dòng, số benign/tấn công với số liệu trên trang UQ** để chứng minh là bản gốc.

- Trang chính thức: https://staff.itee.uq.edu.au/marius/NIDS_datasets/
- Bài báo: M. Luay et al., "Time Matters: Temporal NetFlow Features for ML-Based Network Intrusion Detection," *IEEE Access*, vol. 14, pp. 66899–66913, 2026. https://doi.org/10.1109/ACCESS.2026.3688204 (preprint arXiv:2503.04404)

Với mỗi bộ: in cột và 5 dòng đầu; đếm dòng, benign, tấn công, từng loại; kiểm cột `FLOW_START_MILLISECONDS`, thứ tự thời gian, khoảng bao phủ, phân bố theo ngày; tính SHA-256 file. Kết quả ghi vào `/kaggle/working/<bộ>_check.json`.

In [ ]:
import gc, glob, hashlib, json, os, time
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.csv as pacsv
import pyarrow.compute as pc

# Số liệu trên trang UQ (02/11/2026). Trang tự mâu thuẫn ở 2 bộ, ghi chú kèm theo.
OFFICIAL = {
    "NF-ToN-IoT-v3": {"rows": 27_520_260, "benign": 16_792_214, "attack": 10_728_046},
    "NF-UNSW-NB15-v3": {"rows": 2_365_424, "benign": 2_237_731, "attack": 127_639,
                        "note": "trang ghi attack 127.639 nhưng tổng từng lớp là 127.693 = rows - benign"},
    "NF-BoT-IoT-v3": {"rows": 16_993_808, "benign": 51_989, "attack": 16_881_819,
                      "note": "trang tự mâu thuẫn: benign + attack = 16.933.808 khác rows; tổng từng lớp 16.881.855"},
    "NF-CICIDS2018-v3": {"rows": 20_115_529, "benign": 17_514_626, "attack": 2_600_903},
}
TIME_COLS = ["FLOW_START_MILLISECONDS", "FLOW_END_MILLISECONDS"]

files = sorted(glob.glob("/kaggle/input/**/*.csv", recursive=True))
for f in files:
    print(f"{os.path.getsize(f) / 2**30:7.2f} GiB  {f}")
PATHS = {os.path.basename(f)[:-4]: f for f in files}

In [ ]:
def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(64 << 20), b""):
            h.update(chunk)
    return h.hexdigest()


def check(name):
    path = PATHS[name]
    s = {"dataset": name, "source": "kaggle:athena21/netflow-v3-datasets",
         "official_uq_page": OFFICIAL[name], "file": {"path": path, "bytes": os.path.getsize(path)}}

    head = pd.read_csv(path, nrows=5)
    cols = list(head.columns)
    s["columns"], s["n_columns"] = cols, len(cols)
    s["has_time_cols"] = {c: c in cols for c in TIME_COLS}
    s["head5"] = head.astype(str).to_dict(orient="records")
    print(f"\n===== {name}: {len(cols)} cột; có cột thời gian: {s['has_time_cols']}")
    print(head[[c for c in TIME_COLS + ["IPV4_SRC_ADDR", "IPV4_DST_ADDR", "Attack", "Label"] if c in cols]])

    t0 = time.time()
    tbl = pacsv.read_csv(
        path,
        read_options=pacsv.ReadOptions(block_size=64 << 20),
        convert_options=pacsv.ConvertOptions(
            include_columns=TIME_COLS + ["IPV4_SRC_ADDR", "IPV4_DST_ADDR", "Label", "Attack"],
            column_types={c: pa.int64() for c in TIME_COLS},
        ),
    )
    s["read"] = {"seconds": round(time.time() - t0, 1), "gib_ram": round(tbl.nbytes / 2**30, 2)}

    label = tbl["Label"].to_numpy()
    counts = {"rows": tbl.num_rows, "benign": int((label == 0).sum()), "attack": int((label == 1).sum())}
    s["counts"] = counts
    s["match_official"] = {k: counts[k] == OFFICIAL[name][k] for k in ("rows", "benign", "attack")}
    attack = tbl["Attack"].to_pandas()
    s["per_class"] = {k: int(v) for k, v in attack.value_counts().items()}
    s["n_src_ip"] = int(pc.count_distinct(tbl["IPV4_SRC_ADDR"]).as_py())
    s["n_dst_ip"] = int(pc.count_distinct(tbl["IPV4_DST_ADDR"]).as_py())
    print("Đếm:", counts, "| khớp trang UQ:", s["match_official"])
    print(attack.value_counts().to_string())

    start = tbl["FLOW_START_MILLISECONDS"].to_numpy()
    end = tbl["FLOW_END_MILLISECONDS"].to_numpy()
    d = np.diff(start)
    s["timestamps"] = {
        "null_start": int(tbl["FLOW_START_MILLISECONDS"].null_count),
        "min_start_utc": str(pd.to_datetime(start.min(), unit="ms", utc=True)),
        "max_start_utc": str(pd.to_datetime(start.max(), unit="ms", utc=True)),
        "span_hours": float((start.max() - start.min()) / 3.6e6),
        "frac_nondecreasing": float((d >= 0).mean()),
        "frac_strictly_increasing": float((d > 0).mean()),
        "n_backward_steps": int((d < 0).sum()),
        "max_backward_jump_s": float(-d.min() / 1000) if (d < 0).any() else 0.0,
        "frac_end_ge_start": float((end >= start).mean()),
        "duration_ms_p50": float(np.percentile(end - start, 50)),
        "duration_ms_p99": float(np.percentile(end - start, 99)),
    }
    for k, v in s["timestamps"].items():
        print(f"  {k:24s}: {v}")

    day = pd.to_datetime(start, unit="ms", utc=True).floor("D")
    by_day = pd.DataFrame({"day": day, "attack": attack}).groupby(["day", "attack"]).size().unstack(fill_value=0)
    s["by_day"] = {str(k.date()): {c: int(v) for c, v in row.items() if v} for k, row in by_day.iterrows()}
    with pd.option_context("display.max_columns", None, "display.width", 250):
        print(by_day)

    del tbl, label, attack, start, end, d, day, by_day
    gc.collect()
    t0 = time.time()
    s["file"]["sha256"] = sha256(path)
    print("SHA-256:", s["file"]["sha256"], f"({time.time() - t0:.0f} s)")

    out = f"/kaggle/working/{name}_check.json"
    with open(out, "w", encoding="utf-8") as f:
        json.dump(s, f, ensure_ascii=False, indent=1, default=str)
    return s

In [ ]:
summary = {}
for name in OFFICIAL:
    s = check(name)
    summary[name] = {k: s[k] for k in ("counts", "match_official", "n_src_ip", "n_dst_ip")}
    summary[name]["span_hours"] = s["timestamps"]["span_hours"]
    summary[name]["frac_nondecreasing"] = s["timestamps"]["frac_nondecreasing"]

In [ ]:
print(json.dumps(summary, indent=1))